# 🧪 Teste Simples - Integração API Deepseek

**Objetivo:** Testar a integração com a API Deepseek (via OpenRouter) para obter ingredientes do X-Salada

**Componentes:**
- API: OpenRouter (proxy para Deepseek R1)
- Modelo: `deepseek/deepseek-r1:free`
- Prato: X-Salada
- Formato de resposta: JSON estruturado

## 1️⃣ Importar Bibliotecas

In [1]:
import requests
import json
import time
from datetime import datetime

## 2️⃣ Configurar API Key e Endpoint

In [13]:
# Configurações da API OpenRouter
API_KEY = "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"
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "deepseek/deepseek-r1:free"

# Configurações de timeout e retry
TIMEOUT = 30  # segundos
MAX_RETRIES = 5  # tentativas

print("✅ Configurações carregadas")
print(f"📡 API URL: {API_URL}")
print(f"🤖 Modelo: {MODEL}")
print(f"⏱️ Timeout: {TIMEOUT}s")
print(f"🔄 Max retries: {MAX_RETRIES}")

✅ Configurações carregadas
📡 API URL: https://openrouter.ai/api/v1/chat/completions
🤖 Modelo: deepseek/deepseek-r1:free
⏱️ Timeout: 30s
🔄 Max retries: 5


## 3️⃣ Criar Prompt Estruturado

In [14]:
# Nome do prato a ser consultado
NOME_PRATO = "X-Salada"

# Prompt otimizado para obter ingredientes com alergênicos
prompt = f"""# Contextualização

Você é um chef de cozinha da região metropolitana de São Paulo Brasil e trabalha numa hamburgueria renomada. Seu restaurante tem ótimas recomendações.

# Tarefa

Passe os ingredientes da receita do {NOME_PRATO} que você faz, incluindo informações sobre os possíveis alergênicos.
Traga informações única e exclusivamente dos ingredientes, sem modo de preparo e outras informações.

# Formato da resposta

Responda no seguinte formato JSON:

{{
  "ingredientes": [
    {{
      "nome": "nome do ingrediente",
      "quantidade": "quantidade do ingrediente",
      "unidade": "unidade de medida",
      "alergenico": true/false
    }}
  ]
}}

# Exemplo de resposta

{{
    "ingredientes": [
        {{
            "nome": "Farinha de Trigo",
            "quantidade": "500",
            "unidade": "gramas",
            "alergenico": true
        }},
        {{
            "nome": "Ovos",
            "quantidade": "3",
            "unidade": "unidades",
            "alergenico": true
        }},
        {{
            "nome": "Açúcar",
            "quantidade": "1",
            "unidade": "xícara",
            "alergenico": false
        }}
    ]
}}"""

print(f"📝 Prompt criado para: {NOME_PRATO}")
print(f"📏 Tamanho do prompt: {len(prompt)} caracteres")
print("\n" + "="*80)
print("PROMPT COMPLETO:")
print("="*80)
print(prompt)

📝 Prompt criado para: X-Salada
📏 Tamanho do prompt: 1156 caracteres

PROMPT COMPLETO:
# Contextualização

Você é um chef de cozinha da região metropolitana de São Paulo Brasil e trabalha numa hamburgueria renomada. Seu restaurante tem ótimas recomendações.

# Tarefa

Passe os ingredientes da receita do X-Salada que você faz, incluindo informações sobre os possíveis alergênicos.
Traga informações única e exclusivamente dos ingredientes, sem modo de preparo e outras informações.

# Formato da resposta

Responda no seguinte formato JSON:

{
  "ingredientes": [
    {
      "nome": "nome do ingrediente",
      "quantidade": "quantidade do ingrediente",
      "unidade": "unidade de medida",
      "alergenico": true/false
    }
  ]
}

# Exemplo de resposta

{
    "ingredientes": [
        {
            "nome": "Farinha de Trigo",
            "quantidade": "500",
            "unidade": "gramas",
            "alergenico": true
        },
        {
            "nome": "Ovos",
            "quanti

## 4️⃣ Preparar Payload da Requisição

In [15]:
# Headers da requisição
headers = {
    'Authorization': f'Bearer {API_KEY}',
    'Content-Type': 'application/json',
}

# Payload da requisição
payload = {
    'model': MODEL,
    'messages': [
        {'role': 'user', 'content': prompt}
    ]
}

print("✅ Payload preparado")
print(f"🤖 Modelo: {payload['model']}")
print(f"💬 Mensagens: {len(payload['messages'])}")

✅ Payload preparado
🤖 Modelo: deepseek/deepseek-r1:free
💬 Mensagens: 1


## 5️⃣ Executar Requisição com Retry e Backoff

In [16]:
# Variável para armazenar a resposta
response = None
success = False

print(f"🚀 Iniciando requisição para obter ingredientes de: {NOME_PRATO}")
print(f"⏰ Hora de início: {datetime.now().strftime('%H:%M:%S')}")
print("\n" + "="*80)

for attempt in range(MAX_RETRIES):
    try:
        print(f"\n🔄 Tentativa {attempt + 1}/{MAX_RETRIES}")
        
        # Fazer a requisição
        response = requests.post(API_URL, headers=headers, json=payload, timeout=TIMEOUT)
        
        # Verificar status code
        print(f"📊 Status Code: {response.status_code}")
        
        if response.status_code == 429:
            # Rate limit detectado
            if attempt < MAX_RETRIES - 1:
                wait_time = 10 * (2 ** attempt)  # Backoff exponencial: 10s, 20s, 40s, 80s
                print(f"⚠️ Rate limit detectado! Aguardando {wait_time}s antes de tentar novamente...")
                time.sleep(wait_time)
                continue
            else:
                print("❌ Rate limit persistente após todas as tentativas")
                break
        
        elif response.status_code == 401:
            print("❌ Erro 401: API key inválida")
            break
        
        elif response.status_code == 200:
            print("✅ Requisição bem-sucedida!")
            success = True
            break
        
        else:
            # Outros erros HTTP
            response.raise_for_status()
            break
    
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout após {TIMEOUT}s")
        if attempt < MAX_RETRIES - 1:
            print("🔄 Tentando novamente...")
            time.sleep(5)
        else:
            print("❌ Timeout persistente após todas as tentativas")
    
    except requests.exceptions.RequestException as e:
        print(f"❌ Erro na requisição: {str(e)}")
        break

print("\n" + "="*80)
print(f"⏰ Hora de término: {datetime.now().strftime('%H:%M:%S')}")
print(f"🎯 Status final: {'SUCESSO ✅' if success else 'FALHA ❌'}")

🚀 Iniciando requisição para obter ingredientes de: X-Salada
⏰ Hora de início: 06:54:54


🔄 Tentativa 1/5
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 10s antes de tentar novamente...
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 10s antes de tentar novamente...

🔄 Tentativa 2/5

🔄 Tentativa 2/5
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 20s antes de tentar novamente...
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 20s antes de tentar novamente...

🔄 Tentativa 3/5

🔄 Tentativa 3/5
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 40s antes de tentar novamente...
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 40s antes de tentar novamente...

🔄 Tentativa 4/5

🔄 Tentativa 4/5
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 80s antes de tentar novamente...
📊 Status Code: 429
⚠️ Rate limit detectado! Aguardando 80s antes de tentar novamente...

🔄 Tentativa 5/5

🔄 Tentativa 5/5
📊 Status Code: 429
❌ Rate limit persistente após todas 

## 6️⃣ Processar e Exibir Resposta

In [18]:
if success and response:
    try:
        # Parse da resposta JSON
        response_json = response.json()
        
        print("📦 RESPOSTA COMPLETA DA API:")
        print("="*80)
        print(json.dumps(response_json, indent=2, ensure_ascii=False))
        
        # Extrair o conteúdo da resposta
        if 'choices' in response_json and len(response_json['choices']) > 0:
            content = response_json['choices'][0]['message']['content']
            
            print("\n" + "="*80)
            print(f"🍔 INGREDIENTES DO {NOME_PRATO.upper()}:")
            print("="*80)
            print(content)
            
            # Tentar parsear o JSON dos ingredientes
            try:
                # Remover markdown code blocks se existirem
                if '```json' in content:
                    content = content.split('```json')[1].split('```')[0].strip()
                elif '```' in content:
                    content = content.split('```')[1].split('```')[0].strip()
                
                ingredientes_json = json.loads(content)
                
                print("\n" + "="*80)
                print("📋 INGREDIENTES ESTRUTURADOS:")
                print("="*80)
                
                if 'ingredientes' in ingredientes_json:
                    for i, ing in enumerate(ingredientes_json['ingredientes'], 1):
                        alergenico = "⚠️ ALERGÊNICO" if ing.get('alergenico', False) else "✅ Não alergênico"
                        print(f"\n{i}. {ing.get('nome', 'N/A')}")
                        print(f"   Quantidade: {ing.get('quantidade', 'N/A')} {ing.get('unidade', '')}")
                        print(f"   Status: {alergenico}")
                else:
                    print(json.dumps(ingredientes_json, indent=2, ensure_ascii=False))
            
            except json.JSONDecodeError as e:
                print(f"\n⚠️ Não foi possível parsear JSON dos ingredientes: {str(e)}")
                print("📝 Resposta em formato de texto já exibida acima")
        
        else:
            print("\n⚠️ Estrutura de resposta inesperada")
    
    except Exception as e:
        print(f"\n❌ Erro ao processar resposta: {str(e)}")
        print(f"\n📄 Resposta bruta: {response.text[:500]}...")

else:
    print("\n❌ Não foi possível obter resposta da API")
    if response:
        print(f"\n📄 Resposta do servidor: {response.text[:500]}")


❌ Não foi possível obter resposta da API


## 7️⃣ Estatísticas da Requisição

In [19]:
if success and response:
    print("📊 ESTATÍSTICAS DA REQUISIÇÃO:")
    print("="*80)
    print(f"✅ Status: Sucesso")
    print(f"📡 Status Code: {response.status_code}")
    print(f"⏱️ Tempo de resposta: {response.elapsed.total_seconds():.2f}s")
    print(f"📦 Tamanho da resposta: {len(response.content)} bytes")
    print(f"🔤 Encoding: {response.encoding}")
    
    # Headers de resposta relevantes
    print("\n🔍 Headers relevantes:")
    for header in ['content-type', 'x-ratelimit-limit', 'x-ratelimit-remaining', 'x-ratelimit-reset']:
        if header in response.headers:
            print(f"  • {header}: {response.headers[header]}")
else:
    print("\n❌ Requisição não foi bem-sucedida - sem estatísticas disponíveis")


❌ Requisição não foi bem-sucedida - sem estatísticas disponíveis


## 📝 Conclusão

**Teste concluído!**

Este notebook demonstrou:
- ✅ Integração com API OpenRouter/Deepseek
- ✅ Retry automático com backoff exponencial
- ✅ Tratamento de rate limits (429)
- ✅ Parse de resposta JSON estruturado
- ✅ Identificação de ingredientes alergênicos

**Para testar outro prato:**
1. Volte à célula 3 ("Criar Prompt Estruturado")
2. Altere o valor de `NOME_PRATO`
3. Execute novamente as células a partir da célula 3